In [41]:
#Import Libraries
import pandas as pd
import numpy as np

import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.figure_factory as ff
from matplotlib import pyplot as plt

##Cross-Validation using KFold
from numpy import array
from sklearn.model_selection import KFold

##Biclustering
from sklearn.cluster import SpectralBiclustering

##Scaler
from sklearn.decomposition import PCA
from sklearn import preprocessing
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import Normalizer

In [42]:
#Ingest Data
df = pd.read_csv('Data_PCA.csv')

#Understand Data
print("Data Type Info:")
print(df.info())
print("\n")
print("Summary Info:")
print(df.describe())
print("\n")
print("NaN Count Info:")
df.isnull().sum()

Data Type Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 18 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   GUID      303 non-null    int64 
 1   CARID     303 non-null    object
 2   ATTRIB1   303 non-null    int64 
 3   ATTRIB2   303 non-null    int64 
 4   ATTRIB3   303 non-null    int64 
 5   ATTRIB4   303 non-null    int64 
 6   ATTRIB5   303 non-null    int64 
 7   ATTRIB6   303 non-null    int64 
 8   ATTRIB7   303 non-null    int64 
 9   ATTRIB8   303 non-null    int64 
 10  ATTRIB9   303 non-null    int64 
 11  ATTRIB10  303 non-null    int64 
 12  ATTRIB11  303 non-null    int64 
 13  ATTRIB12  303 non-null    int64 
 14  ATTRIB13  303 non-null    int64 
 15  ATTRIB14  303 non-null    int64 
 16  ATTRIB15  303 non-null    int64 
 17  ATTRIB16  303 non-null    int64 
dtypes: int64(17), object(1)
memory usage: 42.7+ KB
None


Summary Info:
              GUID     ATTRIB1     ATTRIB2    

GUID        0
CARID       0
ATTRIB1     0
ATTRIB2     0
ATTRIB3     0
ATTRIB4     0
ATTRIB5     0
ATTRIB6     0
ATTRIB7     0
ATTRIB8     0
ATTRIB9     0
ATTRIB10    0
ATTRIB11    0
ATTRIB12    0
ATTRIB13    0
ATTRIB14    0
ATTRIB15    0
ATTRIB16    0
dtype: int64

In [43]:
#Understand the Label information in the dataframe.  First two letters indicate country of origin.
Attributes = 16
PCList = []
for i in range(1,Attributes+1):
    PCList.append("PC"+str(i))

df.drop(['GUID'], axis=1, inplace=True)
df.columns = df.columns.str.replace("ATTRIB", "A")
DimensionsList = list(df.select_dtypes("int64").columns)

y = df.CARID  #dependent/label variable
x = pd.DataFrame(df.iloc[:,1:18])  #attribute variables
for colvar in df.columns:
    if df[colvar].dtype=="object":
        print("Column: ", colvar)
        print(df[colvar].value_counts())
        print("\n")

Column:  CARID
JPNLX    31
GERM     31
USACR    31
JPNI     30
USAGP    30
SWEV     30
SWES     30
GERP     30
GERB     30
USAF     30
Name: CARID, dtype: int64




In [44]:
x.corr().style.background_gradient(cmap='inferno')

,A1,A2,A3,A4,A5,A6,A7,A8,A9,A10,A11,A12,A13,A14,A15,A16
A1,1.000000,0.030262,0.450753,0.184831,0.630032,0.751408,0.043388,0.165307,0.826382,-0.148862,0.609976,-0.563978,-0.141478,0.668851,0.636903,-0.344836
A2,0.030262,1.000000,0.401249,-0.069785,0.163508,0.170496,0.455399,-0.002579,0.089523,0.542994,0.230445,0.204616,0.128972,-0.087564,0.220448,0.289795
A3,0.450753,0.401249,1.000000,-0.206306,0.432883,0.629512,0.466402,-0.159487,0.466391,0.259162,0.556222,-0.273670,-0.131654,0.211470,0.671332,-0.153192
A4,0.184831,-0.069785,-0.206306,1.000000,0.319433,0.002529,0.017217,0.791569,0.145498,0.020081,-0.135751,0.164112,0.438970,0.330979,-0.075897,0.173479
A5,0.630032,0.163508,0.432883,0.319433,1.000000,0.589041,0.215997,0.297487,0.624970,0.088179,0.579531,-0.308570,0.069508,0.542938,0.536590,-0.157086
A6,0.751408,0.170496,0.629512,0.002529,0.589041,1.000000,0.232443,0.036225,0.744299,-0.017574,0.670177,-0.516873,-0.134597,0.569015,0.777992,-0.281576
A7,0.043388,0.455399,0.466402,0.017217,0.215997,0.232443,1.000000,0.048288,0.126345,0.581500,0.191454,0.238716,0.248048,-0.121951,0.266452,0.303576
A8,0.165307,-0.002579,-0.159487,0.791569,0.297487,0.036225,0.048288,1.000000,0.153899,0.093349,-0.124737,0.172141,0.460139,0.376673,-0.054916,0.175975
A9,0.826382,0.089523,0.466391,0.145498,0.624970,0.744299,0.126345,0.153899,1.000000,-0.076478,0.616224,-0.538856,-0.105254,0.682661,0.659540,-0.323107
A10,-0.148862,0.542994,0.259162,0.020081,0.088179,-0.017574,0.581500,0.093349,-0.076478,1.000000,0.102282,0.412652,0.304228,-0.221205,0.095057,0.413794


In [53]:
data = pd.DataFrame(x.corr())
model = SpectralBiclustering(n_clusters=6, method='log',
                             random_state=0)
model.fit(data)
fit_data = data[np.argsort(model.row_labels_)]
fit_data = fit_data[:, np.argsort(model.column_labels_)]

KeyError: "None of [Int64Index([2, 8, 11, 6, 3, 12, 9, 7, 4, 10, 13, 0, 15, 5, 14, 1], dtype='int64')] are in the [columns]"

In [21]:
#Corrrelations & Dendrogram Groups
fig = px.imshow(x.corr(), color_continuous_scale = px.colors.sequential.Plasma)
fig.update_layout(width=500, height=500)
fig.show()
figdend = ff.create_dendrogram(x.corr(), labels=DimensionsList)
figdend.update_layout(width=1000, height=500)
figdend.show()

In [22]:
#Based on the dendrogram we create a sort order
AttrSort = [3, 12, 5, 15, 1, 7, 13, 16, 4, 11, 6, 9, 14, 2, 8, 10]

In [23]:
#Correlation Matrix.  Too many variables makes it too messy to read even with the color enhancements.
fig = px.scatter_matrix(
    x,
    dimensions=DimensionsList,
    color=y)
fig.show()

In [24]:
xscale = pd.DataFrame(Normalizer().fit_transform(x))
pca = PCA(n_components = Attributes)
principalComponents = pca.fit_transform(xscale)
principalDataframe = pd.DataFrame(data = principalComponents, columns = PCList)
newDataframe = pd.concat([principalDataframe, y],axis = 1)

In [25]:
#Factor Loadings.  Look for values closer to +1 and -1.  
#Then look for orthogonality between Factors.
#This way the data will cluster on two separate axes more clearly.
Loadings = pd.DataFrame(pca.components_.T * np.sqrt(pca.explained_variance_), columns = PCList)
Loadings.insert(0, "Attribute", DimensionsList)
Loadings.insert(1, "SortOrder", AttrSort)
FactorLoadings = Loadings.sort_values(by='SortOrder')
FactorLoadings.drop(['SortOrder'], axis=1, inplace=True)
FactorLoadings.style.background_gradient(cmap='inferno')

,Attribute,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,PC11,PC12,PC13,PC14,PC15,PC16
4,A5,-0.026547,0.012116,-0.008734,-0.005008,0.014347,-0.001437,-0.024253,-0.014908,0.019391,-0.017545,-0.006176,0.004935,0.011688,0.004544,-0.002160,-0.009423
13,A14,-0.050388,0.040768,0.026009,0.022819,0.001466,-0.015163,0.014149,0.015079,0.018271,-0.001266,0.008740,-0.001868,-0.004713,0.004556,-0.000865,-0.009511
0,A1,-0.057770,0.014916,0.007389,-0.002562,-0.010107,0.015877,-0.014543,0.004768,-0.009839,-0.000583,-0.012990,-0.008750,-0.013432,0.005235,0.018774,-0.007558
8,A9,-0.060369,0.009830,0.009831,0.001390,-0.010332,0.007089,-0.016785,0.018458,-0.012433,0.005856,-0.003209,-0.004426,0.012168,-0.011441,-0.018408,-0.003283
2,A3,-0.030370,-0.037323,-0.021411,-0.014016,-0.015857,-0.002769,0.010127,0.001077,0.020143,-0.005826,-0.008753,-0.022669,-0.007042,-0.003574,-0.007555,0.000471
10,A11,-0.048978,-0.021816,0.006270,0.002611,0.031395,-0.009981,-0.005941,-0.012322,-0.003419,0.009705,-0.003370,0.000112,-0.019604,-0.014839,-0.005901,-0.001139
5,A6,-0.058607,-0.008442,-0.000990,-0.007990,-0.018382,0.001171,0.011546,-0.008192,-0.004826,-0.001252,-0.010775,0.022890,-0.009758,0.012490,-0.012518,-0.002755
14,A15,-0.052252,-0.021132,-0.005986,-0.015865,0.007477,-0.007157,0.021090,-0.010521,-0.016869,0.004351,0.006594,-0.006529,0.014722,-0.000062,0.005395,-0.013365
11,A12,0.099798,-0.012010,0.013977,-0.003822,0.009636,0.017546,0.008252,0.004056,0.010009,0.017652,-0.018218,0.000720,0.002005,0.000698,-0.003995,-0.011009
15,A16,0.070024,-0.012368,0.027117,0.001158,-0.004931,0.002724,0.004052,-0.006164,-0.013648,-0.031265,0.002549,-0.005052,-0.006970,-0.006382,-0.006248,-0.005815


In [26]:
variance = pca.explained_variance_ratio_ #calculate variance ratios
vardf = pd.DataFrame(variance, columns=['Variance']) #cumulative sum of variance explained with [n] features
vardf['PrincipalComponent'] = PCList
vardf['CumulativeVariance'] = vardf['Variance'].cumsum()

In [27]:
#View the covariance matrix.  This shows how the attributes move together (degree of correlation)
covdf = pd.DataFrame(pca.get_covariance())
# covdf.columns = dimensions
# covdf.insert(0, 'Dimensions',dimensions)
covdf.style.background_gradient(cmap='inferno')

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
0,0.005169,-0.001945,0.001097,-0.000399,0.001530,0.003241,-0.001554,-0.000470,0.003908,-0.002637,0.002250,-0.005644,-0.002485,0.003130,0.002293,-0.003877
1,-0.001945,0.004930,0.000532,-0.001850,-0.001145,-0.001126,0.000986,-0.001203,-0.001787,0.001904,-0.000692,0.002202,0.000029,-0.002500,-0.001026,0.001645
2,0.001097,0.000532,0.004483,-0.003136,0.000474,0.002227,0.000749,-0.002604,0.001167,0.000051,0.001724,-0.002691,-0.002100,-0.000330,0.002404,-0.002041
3,-0.000399,-0.001850,-0.003136,0.008468,0.000349,-0.002115,-0.000803,0.005729,-0.001026,-0.001056,-0.002884,0.001441,0.002192,0.000906,-0.002620,0.000335
4,0.001530,-0.001145,0.000474,0.000349,0.002968,0.001183,-0.000739,0.000249,0.001417,-0.001122,0.001272,-0.002906,-0.001137,0.001335,0.000835,-0.002131
5,0.003241,-0.001126,0.002227,-0.002115,0.001183,0.005191,-0.000859,-0.001594,0.003251,-0.002088,0.002597,-0.005613,-0.002772,0.002312,0.003238,-0.003800
6,-0.001554,0.000986,0.000749,-0.000803,-0.000739,-0.000859,0.002998,-0.000952,-0.001370,0.001631,-0.000832,0.001862,0.000500,-0.002399,-0.000589,0.001119
7,-0.000470,-0.001203,-0.002604,0.005729,0.000249,-0.001594,-0.000952,0.007130,-0.000829,-0.000564,-0.002488,0.000880,0.002007,0.001369,-0.002274,0.000223
8,0.003908,-0.001787,0.001167,-0.001026,0.001417,0.003251,-0.001370,-0.000829,0.005466,-0.002424,0.002438,-0.005889,-0.002577,0.003332,0.002557,-0.004069
9,-0.002637,0.001904,0.000051,-0.001056,-0.001122,-0.002088,0.001631,-0.000564,-0.002424,0.004521,-0.000898,0.003375,0.000946,-0.003066,-0.001164,0.002280


In [28]:
#View the Scree plot to identify the elbow in the curve based on the cumulative scores.
fig = make_subplots(rows=2, cols=1)
fig.add_trace(
    go.Bar(x=vardf['PrincipalComponent'], 
           y=vardf['Variance']), row=1, col=1,
)

fig.add_trace(
    go.Scatter(x=vardf['PrincipalComponent'], 
               y=vardf['CumulativeVariance']), row=2, col=1
)

fig.update_layout(height=500, width=1000, title_text="Individual & Cumulative Scree Plots")
fig.show()

In [29]:
#  PC1-PC5 are give the highest returns (81%).

fig1 = px.scatter(
                x = newDataframe.PC1,
                y = newDataframe.PC2,
                color = newDataframe.CARID,
                marginal_y="rug", 
                marginal_x="rug")
fig2 = px.scatter(
                x = newDataframe.PC1,
                y = newDataframe.PC3,
                color = newDataframe.CARID,
                marginal_y="rug", 
                marginal_x="rug")
fig3 = px.scatter(
                x = newDataframe.PC2,
                y = newDataframe.PC3,
                color = newDataframe.CARID,
                marginal_y="histogram", 
                marginal_x="histogram")
fig1.show()
fig2.show()
fig3.show()